In [ ]:
import pandas as pd
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

from src.config import DATA_RAW_DIR, DATA_PROCESSED_DIR

df =pd.read_csv(DATA_RAW_DIR / 'xauusd_daily.csv')
df["datetime"] =pd.to_datetime(df["datetime"])

from src.features.calendar import filter_weekdays
df = filter_weekdays(df)


In [ ]:
print (df.info())

print(df.describe())

#duplikaty

print(f"Duplikaty dat : {df["datetime"].duplicated().sum()}")

date_range = pd.date_range(start = df["datetime"].min(), end = df["datetime"].max())
mising_dates = set(date_range) - set(df["datetime"])

print(f"{len(mising_dates)} missing dates")

missing_weekdays= [d for d in mising_dates if d.weekday()< 5]
print(f"{len(missing_weekdays)} missing weekdays")


print(sorted(missing_weekdays))


In [ ]:
import src.features.indicators as indicators
import importlib
importlib.reload(indicators)

df  = indicators.add_sma(df, "close",window=20)
df = indicators.add_rsi(df, "close",window=14)
df = indicators.add_ema(df, "close",window=20)
df = indicators.add_macd(df,"close")
df = indicators.add_bollinger_bands(df,"close")

df.tail(20)




In [ ]:
print(df.isna().sum())

In [ ]:
import src.features.build_features as build_features_module

importlib.reload(build_features_module)

df = pd.read_csv(DATA_RAW_DIR / "xauusd_daily.csv")
df["datetime"] = pd.to_datetime(df["datetime"])

df = build_features_module.build_features(df)

df.tail(20)
print(df.columns.to_list())

In [ ]:
df = pd.read_csv(DATA_PROCESSED_DIR/"xauusd_daily_features.csv")
df["datetime"] = pd.to_datetime(df["datetime"])

print(df["target"].value_counts())
print(df["target"].value_counts(normalize=True))

In [ ]:
from src.validation.splits import create_time_series_splits

splits = create_time_series_splits(df,n_splits=5)


for split_number ,(train_index, test_index) in enumerate(splits,start=1):
    train_df =df.iloc[train_index]
    test_df = df.iloc[test_index]
    print(f"split number {split_number}")
    print(f"train {train_index[0]}  {train_index[-1]}")
    print(f"train data min {train_df["datetime"].min()} max - {train_df["datetime"].max()}")
    print(f"test {test_index[0]}  {test_index[-1]}")
    print(f"test data min {test_df["datetime"].min()} max - {test_df["datetime"].max()}")
    print(f"{len(train_index)} train and {len(test_index)} test")





In [ ]:
import importlib

from sklearn.metrics import accuracy_score
import src.models.baseline as baseline_module

importlib.reload(baseline_module)


print("Always up baseline")

for split_number, (train_index, test_index) in enumerate(splits, start=1):
    test_df = df.iloc[test_index].copy()

    y_test = test_df["target"]
    y_pred = baseline_module.always_up_baseline(len(test_df))

    accuracy = accuracy_score(y_test, y_pred)

    print(f"Split {split_number}: {accuracy:.4f}")


print("\nPrevious day direction baseline")

for split_number, (train_index, test_index) in enumerate(splits, start=1):
    test_df = df.iloc[test_index].copy()

    y_test = test_df["target"]
    y_pred = baseline_module.previous_day_direction_baseline(test_df)

    accuracy = accuracy_score(y_test, y_pred)

    print(f"Split {split_number}: {accuracy:.4f}")

In [ ]:
df_h5 = pd.read_csv(DATA_PROCESSED_DIR/"xauusd_daily_features_h5.csv")

df_h5["datetime"]= pd.to_datetime(df_h5["datetime"])


splits_h5 = create_time_series_splits(df_h5,n_splits=5)

for split_number, (train_index, test_index) in enumerate(splits_h5):

    test_df = df_h5.iloc[test_index].copy()
    y_test = test_df["target"]
    y_pred = baseline_module.always_up_baseline(len(test_df))
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Split {split_number}: {accuracy:.4f}")


for split_number, (train_index, test_index) in enumerate(splits_h5):

    test_df = df_h5.iloc[test_index].copy()
    y_test = test_df["target"]
    y_pred = baseline_module.previous_day_direction_baseline(test_df)

    accuracy = accuracy_score(y_test, y_pred)
    print(f"Split {split_number}: {accuracy:.4f}")






**Uwaga: opis liczbowy poniżej pochodzi z historycznego kalendarza z weekendami. Aktualne wyniki: WEEKDAY_RESULTS.md.**

## Eksperyment: czy DXY i rentowności poprawiają predykcję?

Sekcja działa niezależnie od wcześniejszych komórek i zapisanych CSV z cechami.
Porównujemy te same obserwacje i podziały chronologiczne dla wszystkich wariantów.
Predykcja po zamknięciu XAU/USD, target: wzrost close za 1 lub 5 **obserwacji**.
Cechy intermarket opóźniamy o jedną obserwację XAU przed usuwaniem braków.
To założenie ostrożnościowe, nie gwarancja dostępności: pliki nie zawierają godzin publikacji
ani historycznych wersji FRED, a broad USD może mieć większe opóźnienie.
Łączenie asof pomija puste wpisy źródeł i bierze ostatnią znaną wartość z przeszłości;
pokazujemy koszt wspólnej próby. Wartość może być nieaktualna przy dłuższej przerwie źródła.
`gap=horizon` oddziela etykiety treningowe od początku testu.
Wyniki są eksploracyjne (bez strojenia), nie stanowią backtestu strategii ani pomiaru zysku.

Rozszerzenie: 10 cech trendu 20/50 obserwacji. DXY: relacje średnich minus 1,
zmienność = std zwrotów 1d z 20 obserwacji (ddof=1, bez annualizacji).
Rentowności: różnice w punktach procentowych, bez dzielenia przez średnią.
Okna 50 obserwacji wymagają rozgrzewki. Dni bez nowego odczytu zachowują ostatnią znaną wartość.
Wyniki poprzedniego uruchomienia nie są bezpośrednio porównywalne po poprawce łączenia braków FRED.
Warianty `+old_*` odtwarzają poprzednie zestawy na tej samej nowej próbie.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score

project_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/config.py').exists())
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from src.config import DATA_RAW_DIR, DATA_PROCESSED_DIR
from src.features.indicators import add_all_indicators
import importlib
import src.features.intermarket as intermarket
importlib.reload(intermarket)
add_intermarket_features = intermarket.add_intermarket_features

raw = pd.read_csv(DATA_RAW_DIR / 'xauusd_daily.csv', parse_dates=['datetime'])
from src.features.calendar import filter_weekdays
raw = filter_weekdays(raw)
assert not raw['datetime'].duplicated().any(), 'Powtórzone daty XAU'
technical = ['SMA_20', 'RSI_14', 'EMA_20', 'MACD', 'MACD_Signal', 'MACD_HIST',
             'BB_UPPER', 'BB_LOWER', 'PRICE_TO_SMA20', 'PRICE_TO_EMA20',
             'BB_WIDTH', 'BB_POSITION', 'RETURN_1D', 'RETURN_5D']
dxy_features = ['dxy_return_1d', 'dxy_return_5d', 'dxy_vs_sma20']
yield_features = ['dgs10_change_1d', 'dgs10_change_5d', 'dfii10_change_1d',
                  'dfii10_change_5d', 'breakeven_10y', 'breakeven_change_1d', 'breakeven_change_5d']
broad_features = ['usd_broad_return_1d', 'usd_broad_return_5d']
old_dxy_features = dxy_features.copy()
old_yield_features = yield_features.copy()
dxy_features += ['dxy_return_20d', 'dxy_vs_sma50', 'dxy_sma20_vs_sma50', 'dxy_volatility_20d']
yield_features += ['dgs10_change_20d', 'dgs10_vs_sma20', 'dgs10_sma20_vs_sma50',
                   'dfii10_change_20d', 'dfii10_vs_sma20', 'dfii10_sma20_vs_sma50']
macro_features = dxy_features + yield_features + broad_features
feature_sets = {
    'technical': technical,
    '+old_DXY': technical + old_dxy_features,
    '+old_yields': technical + old_yield_features,
    '+old_DXY+yields': technical + old_dxy_features + old_yield_features,
    '+old_all': technical + old_dxy_features + old_yield_features + broad_features,
    '+DXY': technical + dxy_features,
    '+yields': technical + yield_features,
    '+DXY+yields': technical + dxy_features + yield_features,
    '+all': technical + macro_features,
}
features = add_intermarket_features(add_all_indicators(raw))
features[macro_features] = features[macro_features].shift(1)
features = features.replace([np.inf, -np.inf], np.nan)
display(features[macro_features].agg(['count', 'mean', 'std', 'min', 'max']).T)
display(features[technical + macro_features].isna().sum().rename('missing_before_drop').to_frame())

In [ ]:
# Diagnostyka źródeł: zakres, braki, duplikaty i koniec pokrycia.
source_quality = []
for filename, date_col, value_col in [
    ('dxy_reconstructed_daily.csv', 'datetime', 'dxy'),
    ('dgs10_daily.csv', 'date', 'value'),
    ('dfii10_daily.csv', 'date', 'value'),
    ('usd_broad_daily.csv', 'date', 'value'),
]:
    source = pd.read_csv(DATA_RAW_DIR / filename, parse_dates=[date_col])
    valid = source.loc[source[value_col].notna(), date_col]
    source_quality.append(dict(source=filename, rows=len(source), missing=source[value_col].isna().sum(),
                               duplicates=source[date_col].duplicated().sum(),
                               first_valid=valid.min(), last_valid=valid.max(),
                               days_to_xau_end=(raw.datetime.max() - valid.max()).days))
display(pd.DataFrame(source_quality))

In [ ]:
models = {
    'LogisticRegression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, random_state=42)),
    'RandomForest': RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced', n_jobs=1),
}
records, coverage, fold_dates = [], [], []
for horizon in (1, 5):
    data = features.copy()
    future = data['close'].shift(-horizon)
    data['target'] = (future > data['close']).where(future.notna())
    data['label_end'] = data['datetime'].shift(-horizon)
    technical_rows = data.dropna(subset=technical + ['target'])
    data = data.dropna(subset=technical + macro_features + ['target']).copy()
    data['target'] = data['target'].astype(int)
    coverage.append(dict(horizon=horizon, technical_rows=len(technical_rows), common_rows=len(data),
                         lost_rows=len(technical_rows)-len(data), start=data.datetime.min(), end=data.datetime.max()))
    for fold, (train_idx, test_idx) in enumerate(TimeSeriesSplit(n_splits=5, gap=horizon).split(data), 1):
        train, test = data.iloc[train_idx], data.iloc[test_idx]
        assert train.label_end.max() < test.datetime.min(), 'Etykiety treningowe nachodzą na test'
        fold_dates.append(dict(horizon=horizon, fold=fold, train_end=train.datetime.max(),
                               last_train_label=train.label_end.max(), test_start=test.datetime.min(),
                               test_end=test.datetime.max(), n_train=len(train), n_test=len(test)))
        def record(model, variant, pred, prob=None):
            records.append(dict(horizon=horizon, fold=fold, model=model, variant=variant,
                                n_test=len(test), accuracy=accuracy_score(test.target, pred),
                                balanced_accuracy=balanced_accuracy_score(test.target, pred),
                                roc_auc=roc_auc_score(test.target, prob) if prob is not None else np.nan))
        record('always_up', 'baseline', np.ones(len(test), dtype=int))
        record('previous_direction', 'baseline', (test.RETURN_1D > 0).astype(int))
        for model_name, prototype in models.items():
            for variant, columns in feature_sets.items():
                fitted = clone(prototype).fit(train[columns], train.target)
                record(model_name, variant, fitted.predict(test[columns]), fitted.predict_proba(test[columns])[:, 1])
results_intermarket = pd.DataFrame(records)
display(pd.DataFrame(coverage))
display(pd.DataFrame(fold_dates))

In [ ]:
summary_intermarket = results_intermarket.groupby(['horizon', 'model', 'variant']).agg(
    accuracy=('accuracy', 'mean'), balanced_accuracy=('balanced_accuracy', 'mean'),
    auc=('roc_auc', 'mean'), accuracy_std=('accuracy', 'std'))
display(summary_intermarket.round(4))
# Różnice liczymy parami względem technicznych na dokładnie tym samym foldzie.
reference = results_intermarket.query("variant == 'technical'")[
    ['horizon', 'fold', 'model', 'accuracy', 'balanced_accuracy']]
paired = results_intermarket[results_intermarket.variant.str.startswith('+')].merge(
    reference, on=['horizon', 'fold', 'model'], suffixes=('', '_technical'), validate='many_to_one')
paired['delta_accuracy_pp'] = 100 * (paired.accuracy - paired.accuracy_technical)
paired['delta_balanced_pp'] = 100 * (paired.balanced_accuracy - paired.balanced_accuracy_technical)
paired['win'] = paired.delta_accuracy_pp > 0
comparison = paired.groupby(['horizon', 'model', 'variant']).agg(
    delta_accuracy_pp=('delta_accuracy_pp', 'mean'), delta_balanced_pp=('delta_balanced_pp', 'mean'),
    improved_folds=('win', 'sum'), worst_fold_pp=('delta_accuracy_pp', 'min'), best_fold_pp=('delta_accuracy_pp', 'max'))
display(comparison.round(3))
display(paired.pivot(index=['horizon', 'model', 'variant'], columns='fold', values='delta_accuracy_pp').round(3))
DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
results_intermarket.to_csv(DATA_PROCESSED_DIR / 'intermarket_exploration_results.csv', index=False)
comparison.to_csv(DATA_PROCESSED_DIR / 'intermarket_exploration_comparison.csv')
# Bezpośredni wpływ nowych 10 cech, przy identycznych datach i modelach.
old_to_new = {'+old_DXY': '+DXY', '+old_yields': '+yields',
              '+old_DXY+yields': '+DXY+yields', '+old_all': '+all'}
old_results = results_intermarket[results_intermarket.variant.isin(old_to_new)].copy()
old_results['variant'] = old_results.variant.map(old_to_new)
new_vs_old = results_intermarket.merge(old_results, on=['horizon', 'fold', 'model', 'variant'],
                                       suffixes=('_new', '_old'), validate='one_to_one')
new_vs_old['delta_accuracy_pp'] = 100 * (new_vs_old.accuracy_new - new_vs_old.accuracy_old)
new_vs_old['delta_balanced_pp'] = 100 * (new_vs_old.balanced_accuracy_new - new_vs_old.balanced_accuracy_old)
new_vs_old['win'] = new_vs_old.delta_accuracy_pp > 0
extension_comparison = new_vs_old.groupby(['horizon', 'model', 'variant']).agg(
    delta_accuracy_pp=('delta_accuracy_pp', 'mean'), delta_balanced_pp=('delta_balanced_pp', 'mean'),
    improved_folds=('win', 'sum'))
display(extension_comparison.round(3))
extension_comparison.to_csv(DATA_PROCESSED_DIR / 'intermarket_extension_comparison.csv')


**Uwaga: opis liczbowy poniżej pochodzi z historycznego kalendarza z weekendami. Aktualne wyniki: WEEKDAY_RESULTS.md.**

### Jak czytać wynik

Dodatnie `delta_accuracy_pp` oznacza poprawę względem tych samych cech technicznych.
Sprawdź także balanced accuracy, baseline „always up” oraz liczbę poprawionych foldów.
Pojedynczy dobry fold lub średnia bliska zeru nie potwierdza stabilnego sygnału.
Horyzont 5 ma nakładające się etykiety, więc obserwacje nie są niezależne.
Po wyborze wariantu potrzebny będzie osobny, niewykorzystany okres testowy oraz kontrola
rzeczywistych opóźnień publikacji. Obecne porównanie ocenia tylko wspólną próbę po odrzuceniu braków.


### Wynik rozszerzenia o 10 cech

Porównanie nowych i starych zestawów na wspólnej próbie po poprawce łączenia FRED:
- H1, regresja +DXY: 51,99% accuracy, +0,56 pp względem starego DXY (3/5 foldów lepsze). Always up: 52,11%.
- H1, RandomForest +DXY+yields: 50,61%, +0,95 pp względem starego zestawu (4/5 foldów lepsze).
- H5: rozszerzenie pogarsza accuracy wszystkich wariantów RandomForest; dla regresji wyniki są mieszane.
- Pełny zestaw nie daje stabilnej poprawy. Nie porównujemy tych liczb bezpośrednio z wcześniejszym przebiegiem na innej próbie.
- Opis dotyczy tego uruchomienia; po zmianie danych należy odczytać aktualne tabele.


**Uwaga: opis liczbowy poniżej pochodzi z historycznego kalendarza z weekendami. Aktualne wyniki: WEEKDAY_RESULTS.md.**

## Korelacje i znaczenie predykcyjne cech

Uruchom najpierw komórkę inicjalizującą eksperyment intermarket powyżej.
Macierz Spearmana mierzy zależności monotoniczne pomiędzy cechami. Korelacje z targetem
liczymy dla **przyszłego** zwrotu i kierunku H1/H5, nie ceny złota w tym samym dniu.
Macierze pełnej próby są opisowe; nie służą do selekcji przed walidacją.

Znaczenie modelowe: na każdym z pięciu foldów testowych przestawiamy bloki po 20 obserwacji
wybranej cechy (5 powtórzeń). Dodatnia zmiana ROC AUC oznacza pogorszenie po przetasowaniu,
więc model korzystał z cechy. Bloki częściowo zachowują zależności czasowe; nie jest to test istotności.
Wspólnie przestawiamy też całe grupy DXY/rentowności, aby ograniczyć maskowanie znaczenia
przez skorelowane zamienniki. Modele używają pełnego zestawu cech, parametrów i opóźnienia
z poprzedniego eksperymentu. Wyniki zależą od modelu i nie dowodzą przyczynowości.

In [ ]:
from IPython.display import HTML
from html import escape

analysis_columns = technical + macro_features
correlation_data = features.dropna(subset=analysis_columns)
correlation_matrix = correlation_data[analysis_columns].corr(method='spearman')

def correlation_html(matrix, title):
    cells = ['<html><meta charset="utf-8"><body>', '<h3>' + escape(title) + '</h3>',
             '<p>Spearman: niebieski = ujemna, czerwony = dodatnia; zakres −1…1.</p>',
             '<div style="overflow:auto;max-height:850px"><table style="border-collapse:collapse;font:11px sans-serif">',
             '<tr><th></th>']
    cells += ['<th style="writing-mode:vertical-rl;max-height:200px;padding:5px">' + escape(c) + '</th>' for c in matrix.columns]
    cells.append('</tr>')
    for label, row in matrix.iterrows():
        cells.append('<tr><th style="text-align:left;white-space:nowrap;padding:4px">' + escape(label) + '</th>')
        for value in row:
            saturation = round(abs(value) * 75) if pd.notna(value) else 0
            hue = 8 if value >= 0 else 215
            color = f'hsl({hue},{saturation}%,{97-abs(value)*35 if pd.notna(value) else 97}%)'
            cells.append(f'<td style="background:{color};padding:4px;text-align:right">{value:.2f}</td>')
        cells.append('</tr>')
    cells.append('</table></div></body></html>')
    return ''.join(cells)

matrix_html = correlation_html(correlation_matrix, 'Korelacja cech — pełna wspólna próba')
display(HTML(matrix_html))
correlation_matrix.to_csv(DATA_PROCESSED_DIR / 'intermarket_feature_correlations.csv')
(DATA_PROCESSED_DIR / 'intermarket_correlation_matrix.html').write_text(matrix_html, encoding='utf-8')
# Najsilniej powiązane pary — kandydaci do sprawdzenia redundancji.
upper = correlation_matrix.where(np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool))
redundant_pairs = upper.stack().rename('spearman').reset_index()
redundant_pairs.columns = ['feature_1', 'feature_2', 'spearman']
redundant_pairs = redundant_pairs.loc[redundant_pairs.spearman.abs() >= .85]
display(redundant_pairs.sort_values('spearman', key=abs, ascending=False).round(3))

analysis_datasets, target_correlations = {}, {}
for horizon in (1, 5):
    sample = features.copy()
    sample['future_return'] = sample.close.shift(-horizon) / sample.close - 1
    sample['target'] = (sample.future_return > 0).where(sample.future_return.notna())
    sample['label_end'] = sample.datetime.shift(-horizon)
    sample = sample.dropna(subset=analysis_columns + ['future_return', 'target']).copy()
    sample['target'] = sample.target.astype(int)
    analysis_datasets[horizon] = sample
    for target in ('future_return', 'target'):
        target_correlations[f'H{horizon}_{target}'] = sample[analysis_columns].corrwith(sample[target], method='spearman')
target_correlations = pd.DataFrame(target_correlations)
display(HTML(correlation_html(target_correlations, 'Cechy a przyszły zwrot / kierunek XAU')))
target_correlations.to_csv(DATA_PROCESSED_DIR / 'intermarket_target_correlations.csv')

In [ ]:
# Stałe parametry, bez strojenia na foldach testowych.
importance_models = {
    'LogisticRegression': make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, random_state=42)),
    'RandomForest': RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced', n_jobs=1),
}
importance_groups = {c: [c] for c in analysis_columns}
importance_groups.update({'GROUP_DXY': dxy_features, 'GROUP_YIELDS': yield_features,
                          'GROUP_BROAD': broad_features, 'GROUP_ALL_MACRO': macro_features})
importance_records, model_scores = [], []
for horizon, sample in analysis_datasets.items():
    for fold, (train_idx, test_idx) in enumerate(TimeSeriesSplit(n_splits=5, gap=horizon).split(sample), 1):
        train, test = sample.iloc[train_idx], sample.iloc[test_idx]
        assert train.label_end.max() < test.datetime.min()
        x_test = test[analysis_columns].to_numpy()
        blocks = [np.arange(start, min(start + 20, len(test))) for start in range(0, len(test), 20)]
        rng = np.random.default_rng(42 + horizon * 100 + fold)
        permutations = [np.concatenate([blocks[i] for i in rng.permutation(len(blocks))]) for _ in range(5)]
        for model_name, prototype in importance_models.items():
            model = clone(prototype).fit(train[analysis_columns].to_numpy(), train.target)
            base_auc = roc_auc_score(test.target, model.predict_proba(x_test)[:, 1])
            model_scores.append(dict(horizon=horizon, fold=fold, model=model_name, roc_auc=base_auc))
            for feature, columns in importance_groups.items():
                positions = [analysis_columns.index(c) for c in columns]
                for repeat, permutation in enumerate(permutations):
                    shuffled = x_test.copy()
                    shuffled[:, positions] = x_test[permutation][:, positions]
                    score = roc_auc_score(test.target, model.predict_proba(shuffled)[:, 1])
                    importance_records.append(dict(horizon=horizon, fold=fold, model=model_name,
                        feature=feature, repeat=repeat, auc_drop_pp=100 * (base_auc - score)))
        print(f'H{horizon}, fold {fold}: gotowe')
importance_results = pd.DataFrame(importance_records)
# Najpierw średnia powtórzeń w foldzie; powtórzenia nie są niezależnymi okresami.
fold_importance = importance_results.groupby(['horizon', 'model', 'feature', 'fold']).auc_drop_pp.mean().reset_index()
fold_importance['positive'] = fold_importance.auc_drop_pp > 0
importance_summary = fold_importance.groupby(['horizon', 'model', 'feature']).agg(
    auc_drop_pp=('auc_drop_pp', 'mean'), fold_std_pp=('auc_drop_pp', 'std'),
    positive_folds=('positive', 'sum'), worst_fold_pp=('auc_drop_pp', 'min'), best_fold_pp=('auc_drop_pp', 'max')).reset_index()
importance_summary['candidate'] = (importance_summary.positive_folds >= 4) & (importance_summary.auc_drop_pp > 0)
importance_results.to_csv(DATA_PROCESSED_DIR / 'intermarket_permutation_importance.csv', index=False)
importance_summary.to_csv(DATA_PROCESSED_DIR / 'intermarket_importance_summary.csv', index=False)
display(pd.DataFrame(model_scores).groupby(['horizon', 'model']).roc_auc.agg(['mean', 'std']).round(4))
for (horizon, model_name), table in importance_summary.groupby(['horizon', 'model']):
    print(f'H{horizon} / {model_name}: top 12 cech; spadek AUC w punktach procentowych')
    display(table[~table.feature.str.startswith('GROUP_')].sort_values('auc_drop_pp', ascending=False).head(12).round(3))
    print('Grupy cech:')
    display(table[table.feature.str.startswith('GROUP_')].sort_values('auc_drop_pp', ascending=False).round(3))

**Uwaga: opis liczbowy poniżej pochodzi z historycznego kalendarza z weekendami. Aktualne wyniki: WEEKDAY_RESULTS.md.**

### Interpretacja

- Duża korelacja pomiędzy cechami sugeruje powielanie informacji, nie przewagę prognostyczną.
- `auc_drop_pp > 0`: model korzysta z danej informacji na testach. Wartość ujemna oznacza,
  że przetasowanie poprawia wynik; zero oznacza brak wykrytej użyteczności w tym modelu.
- `candidate` to wyłącznie filtr eksploracyjny: dodatnia średnia i poprawny znak w co najmniej 4/5 foldów.
  Nie jest to dowód istotności statystycznej. Sprawdź rozrzut, grupy i wynik bazowy modelu.
- Ważna cecha w modelu z AUC bliskim 0,5 nadal nie oznacza skutecznej prognozy.
  Selekcja wymaga późniejszego porównania z usuniętą cechą oraz nietkniętego holdoutu.

### Wnioski z bieżącego uruchomienia

Najmocniejszy sygnał intermarket w regresji H5: `dfii10_vs_sma20` (spadek AUC 1,33 pp;
4/5 dodatnich foldów), następnie `dxy_vs_sma50` (0,76 pp; 4/5) i
`dxy_sma20_vs_sma50` (0,69 pp; 4/5). W RF H5 grupa rentowności jest dodatnia w 5/5 foldów
(0,83 pp). W RF H1 `dgs10_vs_sma20` (0,34 pp) i `dxy_sma20_vs_sma50` (0,31 pp)
są dodatnie w 5/5 foldów, lecz sam model ma AUC 0,493.

Wśród wszystkich cech najwyżej jest `BB_LOWER` w regresji H5 (2,31 pp), ale wynik ma
duży rozrzut między foldami (std 3,04 pp), a cecha jest silnie skorelowana z innymi
poziomami cenowymi. Przetasowanie może tworzyć nierealistyczne kombinacje takich cech.

Żadnej cechy nie uznajemy jeszcze za potwierdzoną przewagę: średnie AUC pełnych modeli
wynoszą 0,493 / 0,493 dla H1 oraz 0,518 / 0,504 dla H5 (regresja / RF).
Korelacje z przyszłym zwrotem mają maksymalne |rho| około 0,05.
Najlepsi kandydaci do osobnego testu usunięcia cechy to realna rentowność względem SMA20
i trend DXY. Nie usuwamy automatycznie pozostałych cech na podstawie tego rankingu.


## Benchmark 36 konfiguracji

Logika znajduje się w `src/models/benchmark.py`. Uruchom z katalogu projektu:
`python -m src.models.benchmark`. Poniżej tylko odczyt zapisanych wyników; brak duplikacji treningu.
ALL obejmuje również surowe poziomy intermarket, których nie było w poprzednim eksperymencie notebooka.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/config.py').exists())
benchmark_dir = root / 'data/processed/benchmark_weekdays'
benchmark_summary = pd.read_csv(benchmark_dir / 'summary.csv')
display(benchmark_summary.round(4))
display(pd.read_csv(benchmark_dir / 'baseline_summary.csv').round(4))
display(pd.read_csv(benchmark_dir / 'feature_comparison.csv').round(3))
display(pd.read_csv(benchmark_dir / 'stability.csv').groupby('horizon', sort=False).head(3).round(4))